# Inception / GoogLeNet — go wider, not just deeper

> Tutorial pair for [`inception.py`](inception.py).

## 1. Intuition
Why pick *one* filter size when you can try several? The **Inception module**
runs 1x1, 3x3, 5x5 convolutions and a pooling branch **in parallel** and
**concatenates** their outputs, letting the network learn at multiple scales at
once. The danger is cost: a 5x5 conv over hundreds of input channels is huge. The
cure is the **1x1 "bottleneck" convolution**, which cheaply shrinks channel depth
*before* the expensive spatial conv.

## 2. Concept (the slide)
- **Multi-branch module:** parallel $1\times1$, $3\times3$, $5\times5$, and a
  $3\times3$ max-pool branch; all padded to the same spatial size and
  **concatenated on the channel axis**.
- **1x1 convolution = a per-pixel fully-connected layer across channels.** It
  mixes channels and can *reduce* their number with almost no spatial cost.
- **Bottleneck placement:** put a $1\times1$ *reduce* before the $3\times3$ and
  $5\times5$ branches, and a $1\times1$ *project* after the pool branch.
- **GoogLeNet** = a stack of these modules + global average pooling head.

## 3. Math — parameter savings from 1x1 bottlenecks

A conv with kernel $k$, $C_{in}$ inputs and $C_{out}$ outputs costs $k^2C_{in}C_{out}$
weights. Take the **5x5 branch** of GoogLeNet's inception (3a): $C_{in}=192$,
$C_{out}=32$.

**Naive** (direct 5x5):
$$5^2 \cdot 192 \cdot 32 = 25\cdot192\cdot32 = 153{,}600 \text{ weights}.$$

**Bottlenecked** (1x1 reduce $192\to16$, then 5x5 on 16 channels):
$$\underbrace{1^2\cdot192\cdot16}_{\text{reduce}=3072}
  + \underbrace{5^2\cdot16\cdot32}_{\text{spatial}=12800}
  = 15{,}872 \text{ weights}.$$

A **$9.7\times$** reduction on that branch alone. Summed over the whole module the
1x1 bottlenecks cut the parameter count from $\approx 393{\rm k}$ to $\approx 163{\rm k}$
— about **58% fewer**, with negligible loss in accuracy. Intuitively the 1x1 conv
projects the $C_{in}$ channels onto a small informative subspace; the costly
$k^2$ spatial mixing then happens in that low-dimensional space.

## 4. Key building block — the Inception module (with 1x1 bottlenecks)

In [ ]:
# ===== actual implementation from inception.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def conv_params(k: int, c_in: int, c_out: int) -> int:
    """Weight count of one conv layer (no bias): k*k*c_in*c_out."""
    return k * k * c_in * c_out

def inception_param_comparison(c_in: int = 192, reduce_3: int = 96,
                               reduce_5: int = 16,
                               out_1: int = 64, out_3: int = 128,
                               out_5: int = 32, out_pool: int = 32) -> dict:
    r"""Parameters of one Inception module with vs without 1x1 bottlenecks.

    Default channel counts are GoogLeNet's "inception (3a)" module. The 5x5 and
    3x3 branches are the expensive ones; a 1x1 conv first squeezes c_in down to a
    small `reduce` width, so the spatial conv runs in a cheap subspace.

    naive 5x5 branch:        5*5 * c_in * out_5
    bottlenecked 5x5 branch: 1*1 * c_in * reduce_5  +  5*5 * reduce_5 * out_5
    """
    naive = {
        "1x1":  conv_params(1, c_in, out_1),
        "3x3":  conv_params(3, c_in, out_3),
        "5x5":  conv_params(5, c_in, out_5),
        "pool_proj": conv_params(1, c_in, out_pool),
    }
    reduced = {
        "1x1":  conv_params(1, c_in, out_1),
        "3x3_reduce": conv_params(1, c_in, reduce_3),
        "3x3":  conv_params(3, reduce_3, out_3),
        "5x5_reduce": conv_params(1, c_in, reduce_5),
        "5x5":  conv_params(5, reduce_5, out_5),
        "pool_proj": conv_params(1, c_in, out_pool),
    }
    return {
        "naive_total": sum(naive.values()),
        "reduced_total": sum(reduced.values()),
        "naive": naive,
        "reduced": reduced,
    }

import torch

import torch.nn as nn

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def conv_bn_relu(in_c: int, out_c: int, k: int, padding: int = 0) -> nn.Sequential:
    """Conv -> BN -> ReLU, the standard GoogLeNet basic conv unit."""
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, k, padding=padding, bias=False),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
    )

class NaiveInception(nn.Module):
    """Parallel 1x1 / 3x3 / 5x5 / (3x3 max-pool) branches, concatenated on the
    channel axis. All branches use padding so spatial size is preserved, so the
    concatenation aligns. This version has NO bottleneck -> expensive."""

    def __init__(self, in_c: int, out_1: int, out_3: int, out_5: int,
                 out_pool: int):
        super().__init__()
        self.b1 = conv_bn_relu(in_c, out_1, 1)
        self.b3 = conv_bn_relu(in_c, out_3, 3, padding=1)
        self.b5 = conv_bn_relu(in_c, out_5, 5, padding=2)
        self.bpool = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            conv_bn_relu(in_c, out_pool, 1),
        )
        self.out_channels = out_1 + out_3 + out_5 + out_pool

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bpool(x)], dim=1)

class Inception(nn.Module):
    """GoogLeNet Inception module WITH 1x1 dimension-reduction bottlenecks.

    The 3x3 and 5x5 branches first squeeze the input channels with a cheap 1x1
    conv (`reduce_3`, `reduce_5`) before the expensive spatial conv. The pool
    branch projects with a 1x1 too. Output channels = out_1+out_3+out_5+out_pool.
    """

    def __init__(self, in_c: int, out_1: int, reduce_3: int, out_3: int,
                 reduce_5: int, out_5: int, out_pool: int):
        super().__init__()
        self.b1 = conv_bn_relu(in_c, out_1, 1)
        self.b3 = nn.Sequential(
            conv_bn_relu(in_c, reduce_3, 1),          # 1x1 bottleneck (reduce)
            conv_bn_relu(reduce_3, out_3, 3, padding=1),
        )
        self.b5 = nn.Sequential(
            conv_bn_relu(in_c, reduce_5, 1),          # 1x1 bottleneck (reduce)
            conv_bn_relu(reduce_5, out_5, 5, padding=2),
        )
        self.bpool = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            conv_bn_relu(in_c, out_pool, 1),          # 1x1 projection
        )
        self.out_channels = out_1 + out_3 + out_5 + out_pool

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bpool(x)], dim=1)

## 5. Full architecture (PyTorch) — a tiny GoogLeNet

In [ ]:
# ===== actual implementation from inception.py =====
class TinyGoogLeNet(nn.Module):
    """A miniature GoogLeNet: a conv stem, two Inception modules, global average
    pooling, and a linear classifier. Channel counts shrunk so it runs on tiny
    inputs in seconds."""

    def __init__(self, in_c: int = 1, n_classes: int = 10):
        super().__init__()
        self.stem = conv_bn_relu(in_c, 16, 3, padding=1)
        # tiny analogues of inception (3a)/(3b): in=16 -> 32 -> 48 channels
        self.inc1 = Inception(16, out_1=8, reduce_3=8, out_3=12,
                              reduce_5=2, out_5=4, out_pool=8)   # 8+12+4+8 = 32
        self.inc2 = Inception(self.inc1.out_channels,
                              out_1=12, reduce_3=12, out_3=16,
                              reduce_5=4, out_5=8, out_pool=12)   # 12+16+8+12 = 48
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(self.inc2.out_channels, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.inc1(x)
        x = self.inc2(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

def demo():
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.set_num_threads(1)        # tiny ops: avoid thread-thrashing on big CPUs
    dev = get_device()

    # --- (a) The 1x1-bottleneck parameter saving (GoogLeNet inception 3a) -----
    cmp = inception_param_comparison()
    print("Inception module (3a) parameters, naive vs 1x1-bottlenecked:")
    print(f"  naive   (no 1x1 reduce): {cmp['naive_total']:>10,} weights")
    print(f"  reduced (with 1x1)     : {cmp['reduced_total']:>10,} weights")
    print(f"  -> bottlenecks cut params to "
          f"{cmp['reduced_total'] / cmp['naive_total']:.0%} "
          f"({1 - cmp['reduced_total'] / cmp['naive_total']:.0%} fewer).")

    # --- (b) Naive vs bottlenecked module on a tiny input: shapes + params ----
    x = torch.randn(8, 16, 16, 16, device=dev)
    naive = NaiveInception(16, out_1=8, out_3=12, out_5=4, out_pool=8).to(dev)
    redu = Inception(16, out_1=8, reduce_3=8, out_3=12,
                     reduce_5=2, out_5=4, out_pool=8).to(dev)
    pn = sum(p.numel() for p in naive.parameters())
    pr = sum(p.numel() for p in redu.parameters())
    print(f"\nTiny module on {tuple(x.shape)}:")
    print(f"  NaiveInception -> {tuple(naive(x).shape)}, params = {pn:,}")
    print(f"  Inception(1x1) -> {tuple(redu(x).shape)}, params = {pr:,}")

    # --- (c) Build TinyGoogLeNet, count params, train a few steps -------------
    x1 = torch.randn(8, 1, 16, 16, device=dev)
    yb = torch.randint(0, 10, (8,), device=dev)
    net = TinyGoogLeNet(in_c=1, n_classes=10).to(dev)
    out = net(x1)
    n_params = sum(p.numel() for p in net.parameters())
    print(f"\nTinyGoogLeNet: {tuple(x1.shape)} -> {tuple(out.shape)}, "
          f"params = {n_params:,}")
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for _ in range(20):
        opt.zero_grad()
        loss = loss_fn(net(x1), yb)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    print(f"  training loss: {losses[0]:.3f} -> {losses[-1]:.3f} (going down)")

## 6. Run — the bottleneck saving, naive vs reduced shapes, a few training steps

In [ ]:
demo()

## 7. Visualization — where the parameters go, naive vs 1x1-bottlenecked

Per-branch parameter counts for one Inception (3a) module. The naive 3x3 and 5x5
branches dominate; adding cheap 1x1 reduces collapses them dramatically.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import inception as M

cmp = M.inception_param_comparison()
naive, reduced = cmp["naive"], cmp["reduced"]
keys = ["1x1", "3x3", "5x5", "pool_proj"]
# fold the *_reduce costs into their branch for a fair side-by-side
red_branch = {
    "1x1": reduced["1x1"],
    "3x3": reduced["3x3_reduce"] + reduced["3x3"],
    "5x5": reduced["5x5_reduce"] + reduced["5x5"],
    "pool_proj": reduced["pool_proj"],
}
x = np.arange(len(keys)); w = 0.38
plt.figure(figsize=(8, 4.2))
plt.bar(x - w/2, [naive[k] for k in keys], w, label=f"naive ({cmp['naive_total']:,})")
plt.bar(x + w/2, [red_branch[k] for k in keys], w, label=f"with 1x1 ({cmp['reduced_total']:,})")
plt.xticks(x, keys); plt.ylabel("parameters (weights)")
plt.title("Inception (3a): 1x1 bottlenecks shrink the 3x3 & 5x5 branches")
plt.legend(); plt.grid(True, axis="y", alpha=.3); plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Width + multi-scale:** branch concatenation lets one module see several
  receptive fields; the network picks the mix.
- **1x1 convs are the workhorse:** cheap channel mixing / dimensionality
  reduction. They reappear in ResNet bottlenecks and MobileNet pointwise convs.
- **Pitfall — alignment:** every branch must emit the *same spatial size* (use
  padding) so the channel-concatenation lines up.
- **Pitfall — channel budget:** the reduce widths (`reduce_3`, `reduce_5`) are
  hyperparameters; too small and you bottleneck information, too large and you
  lose the savings.